# Unit 1: OpenBCI & BrainFlow 入门

## 学习目标
- 理解脑机接口（BCI）的基本概念
- 了解 OpenBCI 硬件生态
- 认识 BrainFlow 框架的三大核心模块
- 完成环境安装与验证

## 1.1 什么是脑机接口（BCI）？

脑机接口（Brain-Computer Interface, BCI）是一种直接在大脑与外部设备之间建立通信通路的技术。

### 常见信号类型
| 信号类型 | 频率范围 | 典型含义 |
|----------|----------|----------|
| **Delta (δ)** | 0.5 - 4 Hz | 深度睡眠 |
| **Theta (θ)** | 4 - 8 Hz | 冥想、困倦 |
| **Alpha (α)** | 8 - 13 Hz | 放松、闭眼 |
| **Beta (β)** | 13 - 30 Hz | 活跃思考、专注 |
| **Gamma (γ)** | 30 - 50 Hz | 高级认知加工 |

### EEG 信号特点
- 微弱：通常在 10-100 μV（微伏）级别
- 易受干扰：工频噪声（50/60Hz）、眼电、肌电
- 需要放大、滤波、模数转换才能被计算机处理

## 1.2 OpenBCI 硬件概览

OpenBCI 是一个开源的脑机接口硬件平台，提供多种设备：

| 设备 | 通道数 | 采样率 | 连接方式 |
|------|--------|--------|----------|
| **Cyton** | 8 通道 | 250 Hz | USB Dongle (串口) |
| **Cyton + Daisy** | 16 通道 | 125 Hz | USB Dongle (串口) |
| **Ganglion** | 4 通道 | 200 Hz | 蓝牙 / USB Dongle |
| **WiFi Shield** | 继承主控板 | 继承主控板 | WiFi (TCP/UDP) |

本教程使用 BrainFlow 内置的 Synthetic Board（合成板），无需真实硬件。

## 1.3 BrainFlow 框架架构

BrainFlow 是一个跨语言、跨平台的生物传感器数据采集与处理库。

```text
┌─────────────────────────────────────────────────┐
│                  BrainFlow API                   │
│  Python | C++ | Java | C# | R | Julia | Rust    │
├─────────────────────┬───────────────────────────┤
│    BoardShim        │       DataFilter          │
│    (数据采集层)      │       (信号处理层)          │
│  ┌───────────────┐  │  ┌─────────────────────┐  │
│  │ 板卡抽象层     │  │  │ • 数字滤波           │  │
│  │ 统一数据格式   │  │  │ • FFT / PSD          │  │
│  │ 流式传输      │  │  │ • 小波变换           │  │
│  │ 回放与合成    │  │  │ • 频带功率计算        │  │
│  └───────────────┘  │  │ • 特征提取           │  │
│                     │  └─────────────────────┘  │
├─────────────────────┴───────────────────────────┤
│                   MLModel                        │
│  ┌─────────────────────────────────────────┐    │
│  │ • 预训练模型：放松度、专注度、正念度       │    │
│  │ • ONNX 自定义模型支持                     │    │
│  └─────────────────────────────────────────┘    │
└─────────────────────────────────────────────────┘
```

### 核心设计理念

1. **统一数据采集 API**：切换硬件只需修改 `board_id` 和连接参数，其余代码不变
2. **模块化**：三大模块独立，可按需组合使用
3. **跨语言绑定**：Python、C++、Java、C#、R、Julia、Rust、TypeScript 等 8 种语言

## 1.4 环境安装与验证

In [ ]:
# 检查 Python 版本（需要 >= 3.7）
import sys
print(f"Python 版本: {sys.version}")
assert sys.version_info >= (3, 7), "需要 Python 3.7 或更高版本"

In [ ]:
# 导入 BrainFlow 核心模块
import brainflow
import numpy as np

from brainflow.board_shim import (
    BoardShim,          # 板卡控制类
    BrainFlowInputParams, # 连接参数
    BoardIds,           # 板卡 ID 枚举
    BrainFlowError,     # 异常类
    LogLevels,          # 日志级别
    BrainFlowPresets    # 数据预设
)
from brainflow.data_filter import (
    DataFilter,         # 信号处理类
    FilterTypes,        # 滤波器类型
    AggOperations,      # 聚合操作
    DetrendOperations,  # 去趋势操作
    WindowOperations,   # 窗口操作
    NoiseTypes          # 噪声类型
)
from brainflow.ml_model import (
    MLModel,            # 机器学习模型
    BrainFlowMetrics,   # 评估指标
    BrainFlowClassifiers, # 分类器类型
    BrainFlowModelParams  # 模型参数
)

print("✅ 所有 BrainFlow 模块导入成功")

In [ ]:
# 检查可视化库
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

print(f"BrainFlow 已安装")
print(f"NumPy 版本: {np.__version__}")
print(f"Matplotlib 版本: {matplotlib.__version__}")
print(f"Pandas 版本: {pd.__version__}")

# 设置 matplotlib 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("\n✅ 环境验证完成！")

## 1.5 第一个 BrainFlow 程序

使用 Synthetic Board（合成板）创建一个最小的数据采集示例。
合成板生成模拟信号，非常适合同步学习和测试。

In [ ]:
import time

# Step 1: 创建合成板 —— 无需任何硬件
params = BrainFlowInputParams()
board_id = BoardIds.SYNTHETIC_BOARD

board = BoardShim(board_id, params)
print(f"板卡创建成功，板卡类型: Synthetic Board")

# Step 2: 准备会话
board.prepare_session()
print("会话已准备")

# Step 3: 开始数据流
board.start_stream()
print("数据流已启动...")

# Step 4: 采集 5 秒数据
time.sleep(5)
print("采集完成")

# Step 5: 获取数据
data = board.get_board_data()
print(f"获取数据形状: {data.shape}")
print(f"  - 行（通道数）: {data.shape[0]}")
print(f"  - 列（采样点数）: {data.shape[1]}")

# Step 6: 停止并释放
board.stop_stream()
board.release_session()
print("会话已释放")

# 查看前几个数据点
print(f"\n数据样例（前5列）:\n{data[:, :5]}")

## 1.6 核心概念速查

### BoardShim 会话生命周期

```text
BoardShim(board_id, params)    # 1. 创建实例
        │
prepare_session()              # 2. 准备会话（初始化内部数据结构）
        │
start_stream(buffer_size)      # 3. 开始采集（创建环形缓冲区）
        │
        ├─ get_board_data()        # 获取全部数据（清空缓冲区）
        ├─ get_current_board_data(n) # 获取最新 n 个样本（保留缓冲区）
        └─ insert_marker(value)    # 插入事件标记
        │
stop_stream()                  # 4. 停止采集
        │
release_session()              # 5. 释放资源
```

### 数据格式

`get_board_data()` 返回一个 **2D NumPy 数组**：
- **行（row）** = 通道（e.g. 第1行可能是包序号，第2-9行是 EEG 数据）
- **列（col）** = 采样点（时间序列）

使用 `BoardShim.get_board_descr(board_id)` 可以获取每个通道的具体含义。

### 下一篇

→ [Unit 2: BoardShim 基础](unit2_boardshim_basics.ipynb) — 深入理解数据采集与通道管理